In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score, mean_squared_error
import xgboost as xgb
import lightgbm as lgb

In [ ]:
def add_features(data):
    data = data.copy()
    data['Sample Date'] = pd.to_datetime(data['Sample Date'], format='%d-%m-%Y', errors='coerce')
    data = data.dropna(subset=['Sample Date'])
    
    data['lat'] = data['Latitude']
    data['lon'] = data['Longitude']
    
    for angle, prefix in [(data['lat'], 'lat'), (data['lon'], 'lon')]:
        rad = np.radians(angle)
        for k in [1, 2, 3, 4]:
            data[f'{prefix}_sin_{k}'] = np.sin(k * rad)
            data[f'{prefix}_cos_{k}'] = np.cos(k * rad)
    
    data['year'] = data['Sample Date'].dt.year.astype(float)
    data['year_norm'] = (data['year'] - 2011) / 4.0
    data['month'] = data['Sample Date'].dt.month.astype(float)
    data['doy']   = data['Sample Date'].dt.dayofyear.astype(float)
    
    data['doy_sin_365'] = np.sin(2 * np.pi * data['doy'] / 365.25)
    data['doy_cos_365'] = np.cos(2 * np.pi * data['doy'] / 365.25)
    data['month_sin_12'] = np.sin(2 * np.pi * data['month'] / 12)
    data['month_cos_12'] = np.cos(2 * np.pi * data['month'] / 12)
    
    for period, label in [(182.625, '182_6'), (91.3125, '91_3'), (30.4375, '30_4')]:
        data[f'doy_sin_{label}'] = np.sin(2 * np.pi * data['doy'] / period)
        data[f'doy_cos_{label}'] = np.cos(2 * np.pi * data['doy'] / period)
    
    data['lat_doy_sin'] = data['lat_sin_1'] * data['doy_sin_365']
    data['lon_doy_sin'] = data['lon_sin_1'] * data['doy_sin_365']
    data['lat_year']    = data['lat'] * data['year_norm']
    data['lon_year']    = data['lon'] * data['year_norm']
    
    return data

In [ ]:
print("=== Loading & merging training data ===")

df_target = pd.read_csv('water_quality_training_dataset.csv')
df_terra = pd.read_csv('terraclimate_features_training.csv')

# All Landsat data
df_landsat1 = pd.read_csv('landsat_features_training.csv')
df_landsat2 = pd.read_csv('landsat_features_2000_2006_200_samples.csv')
df_landsat3 = pd.read_csv('landsat_features_2015_2023_200_samples.csv')
df_landsat = pd.concat([df_landsat1, df_landsat2, df_landsat3], ignore_index=True)

# Deltares data
df_deltares = pd.read_csv('deltares_water_2010_2015_200_samples.csv')

df_train = df_target.merge(df_terra, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
df_train = df_train.merge(df_landsat, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
df_train = df_train.merge(df_deltares, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
df_train = add_features(df_train)

In [ ]:
raw_features = ['pet', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI',
                'P', 'ETa', 'PET', 'Melt', 'Snow', 'Temp', 'P_res', 'S_res', 
                'Ea_res', 'Qin_res', 'FracFull', 'Qout_res']

engineered = [c for c in df_train.columns if any(x in c for x in [
    'sin', 'cos', 'year', 'doy_', 'month_', 'lat_', 'lon_'
])]

features = raw_features + engineered

print(f"Using {len(features)} features")

In [ ]:
df_train = df_train.fillna(df_train.median(numeric_only=True))  # Use median impute as in benchmark
X = df_train[features]
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

targets = {
    'TA':  df_train['Total Alkalinity'],
    'EC':  df_train['Electrical Conductance'],
    'DRP': df_train['Dissolved Reactive Phosphorus']
}

In [ ]:
xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'learning_rate': 0.01,
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'tree_method': 'hist',
    'seed': 42
}

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.01,
    'max_depth': 5,
    'num_leaves': 31,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'n_jobs': -1,
    'verbosity': -1,
    'random_state': 42,
    'force_col_wise': True
}


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
models = {}

for name, y in targets.items():
    print(f"\n=== Training {name} ===")
    oof = np.zeros(len(y))
    xgb_list = []
    lgb_list = []
    
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_scaled)):
        X_tr, X_val = X_scaled[tr_idx], X_scaled[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        # XGBoost
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)
        evals = [(dval, 'eval')]
        xgb_booster = xgb.train(
            xgb_params,
            dtrain,
            num_boost_round=1000,
            evals=evals,
            early_stopping_rounds=50,
            verbose_eval=100
        )
        xgb_list.append(xgb_booster)
        
        # LightGBM
        lgb_model = lgb.LGBMRegressor(**lgb_params)
        lgb_model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(period=100)]
        )
        lgb_list.append(lgb_model)
        
        # OOF for blend
        xgb_pred = xgb_booster.predict(dval)
        lgb_pred = lgb_model.predict(X_val)
        blend_pred = (xgb_pred + lgb_pred) / 2
        oof[val_idx] = blend_pred
    
    r2 = r2_score(y, oof)
    rmse = np.sqrt(mean_squared_error(y, oof))
    print(f"{name}  CV R² = {r2:.4f}   RMSE = {rmse:.2f}")
    
    models[name] = (xgb_list, lgb_list)

In [ ]:

# ====================== SUBMISSION ======================
print("\n=== Generating submission ===")
sub = pd.read_csv('submission_template.csv')

df_terra_val = pd.read_csv('terraclimate_features_validation.csv')
df_landsat_val = pd.read_csv('landsat_features_validation.csv')

sub = sub.merge(df_terra_val, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
sub = sub.merge(df_landsat_val, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
sub = add_features(sub)

# Add missing Deltares columns as 0
deltares_cols = ['P', 'ETa', 'PET', 'Melt', 'Snow', 'Temp', 'P_res', 'S_res', 
                 'Ea_res', 'Qin_res', 'FracFull', 'Qout_res']
for col in deltares_cols:
    if col not in sub.columns:
        sub[col] = 0

sub = sub.fillna(sub.median(numeric_only=True))  # Median impute for sub

sub_X = sub[features]
sub_X_scaled = scaler.transform(sub_X)
dsub = xgb.DMatrix(sub_X_scaled)

def blend_predict(xgb_models, lgb_models, X, dX):
    xgb_preds = np.mean([m.predict(dX) for m in xgb_models], axis=0)
    lgb_preds = np.mean([m.predict(X) for m in lgb_models], axis=0)
    return (xgb_preds + lgb_preds) / 2

sub['Total Alkalinity']              = blend_predict(models['TA'][0], models['TA'][1], sub_X_scaled, dsub)
sub['Electrical Conductance']        = blend_predict(models['EC'][0], models['EC'][1], sub_X_scaled, dsub)
sub['Dissolved Reactive Phosphorus'] = blend_predict(models['DRP'][0], models['DRP'][1], sub_X_scaled, dsub)

# Ensemble with previous submission
previous_sub = pd.read_csv('submission_larger_landsat_deltares_xgb.csv')
previous_sub['Sample Date'] = pd.to_datetime(previous_sub['Sample Date'])
sub['Sample Date'] = pd.to_datetime(sub['Sample Date'])
sub = sub.merge(previous_sub, on=['Latitude', 'Longitude', 'Sample Date'], how='left', suffixes=('', '_prev'))

sub['Total Alkalinity'] = (sub['Total Alkalinity'] + sub['Total Alkalinity_prev']) / 2
sub['Electrical Conductance'] = (sub['Electrical Conductance'] + sub['Electrical Conductance_prev']) / 2
sub['Dissolved Reactive Phosphorus'] = (sub['Dissolved Reactive Phosphorus'] + sub['Dissolved Reactive Phosphorus_prev']) / 2

sub = sub.drop(columns=[col for col in sub.columns if '_prev' in col])

# Clip
sub['Total Alkalinity'] = sub['Total Alkalinity'].clip(5, 400)
sub['Electrical Conductance'] = sub['Electrical Conductance'].clip(30, 2200)
sub['Dissolved Reactive Phosphorus'] = sub['Dissolved Reactive Phosphorus'].clip(0.5, 250)

final_sub = sub[['Latitude', 'Longitude', 'Sample Date',
                 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

final_sub.to_csv('submission_all_files_blend_ensemble.csv', index=False)
print("✅ Saved → submission_all_files_blend_ensemble.csv")